In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from utils import WBCTestDataset

## Extract features using model from WEIGHTS_PATH

In [ ]:
# =========================
# CONFIG
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHTS_PATH = "resnet_wbc_finetuned_split.pth"
BATCH_SIZE = 128

# =========================
# REBUILD MODEL
# =========================
model = models.resnet50(weights=None)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 13)

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# Remove classifier to get embeddings
model.fc = nn.Identity()

# =========================
# FEATURE EXTRACTION FUNCTION
# =========================
def extract_features(model, dataloader, device):
    all_features = []

    model.eval()
    with torch.no_grad():
        for images in dataloader:
            images = images.to(device)
            outputs = model(images)
            all_features.append(outputs.cpu())

    return torch.cat(all_features).numpy()

# =========================
# CREATE DATASETS
# =========================
train_dataset = WBCTestDataset("train_split.csv", "train")
val_dataset   = WBCTestDataset("val_split.csv", "train")
test_dataset  = WBCTestDataset("test_metadata.csv", "test")

# =========================
# CREATE DATALOADERS
# =========================
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# =========================
# EXTRACT FEATURES
# =========================
print("Extracting train features...")
train_features = extract_features(model, train_loader, DEVICE)

print("Extracting val features...")
val_features = extract_features(model, val_loader, DEVICE)

print("Extracting test features...")
test_features = extract_features(model, test_loader, DEVICE)

# =========================
# SAVE FEATURES
# =========================
base_name = WEIGHTS_PATH.split('.')[0]

np.save(f"train_features_{base_name}.npy", train_features)
np.save(f"val_features_{base_name}.npy", val_features)
np.save(f"test_features_{base_name}.npy", test_features)

print("Done.")
print("Train feature shape:", train_features.shape)
print("Val feature shape:", val_features.shape)
print("Test feature shape:", test_features.shape)

C:\Users\fedem\AppData\Local\Temp\ipykernel_67672\2665284700.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(WEIGHTS_PATH, map_location

Extracting train features...
Extracting val features...
Extracting test features...
Done.
Train feature shape: (24565, 2048)
Val feature shape: (4336, 2048)
Test feature shape: (9634, 2048)


## ResNet submission

In [8]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import cv2
import os
from torchvision import models
from torch.utils.data import Dataset, DataLoader

# ======================
# CONFIG
# ======================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHTS_PATH = "resnet_wbc_augmented_best.pth"
BATCH_SIZE = 128

# ======================
# REBUILD MODEL
# ======================

model = models.resnet50(weights=None)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 13)

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# ======================
# GET CLASS ORDER
# ======================

train_df = pd.read_csv("train_split.csv")
classes = sorted(train_df["label"].unique())
idx_to_class = {i: cls for i, cls in enumerate(classes)}

print("Class order:", idx_to_class)

# ======================
# TEST DATASET
# ======================

class TestDataset(Dataset):
    def __init__(self, csv_file, img_dir):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["ID"]
        img_path = os.path.join(self.img_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))
        image = torch.tensor(image, dtype=torch.float32)

        image = (image - self.mean) / self.std

        return image

# ======================
# LOAD TEST DATA
# ======================

test_dataset = TestDataset("test_metadata.csv", "test")
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# ======================
# INFERENCE
# ======================

all_preds = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())

# Map indices to labels
labels_str = [idx_to_class[i] for i in all_preds]

# ======================
# CREATE SUBMISSION
# ======================

test_df = pd.read_csv("test_metadata.csv")

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "label": labels_str
})

submission.to_csv("submission_resnet.csv", index=False)

print("ResNet submission saved.")

C:\Users\fedem\AppData\Local\Temp\ipykernel_38956\2975910165.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(WEIGHTS_PATH, map_location

Class order: {0: 'BA', 1: 'BL', 2: 'BNE', 3: 'EO', 4: 'LY', 5: 'MMY', 6: 'MO', 7: 'MY', 8: 'PC', 9: 'PLY', 10: 'PMY', 11: 'SNE', 12: 'VLY'}
ResNet submission saved.
